In [ ]:
!pip install -q ultralytics roboflow

!pip install roboflow

import os
from roboflow import Roboflow
rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY", "TU_ROBOFLOW_API_KEY"))
project = rf.workspace("counting-ebsfi").project("vehicle-counting-e6o3d")
version = project.version(20)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to vehicle-counting-20 in yolov11:: 100%|██████████| 13069/13069 [00:02<00:00, 5344.98it/s]


In [ ]:
import yaml
from pathlib import Path

# Detecta la carpeta donde Roboflow descargó el dataset
yamls = [p for p in Path("/content").rglob("data.yaml") if "sample_data" not in str(p)]
DATASET_DIR = yamls[0].parent
print("Dataset encontrado en:", DATASET_DIR)

TRADUCCION = {
    "bike": 0, "motorcycle": 0, "motorbike": 0, "moto": 0,
    "auto": 1, "car": 1, "jeep": 1, "suv": 1, "pickup": 1,
    "minibus": 2, "van": 2,
    "bus": 3,
    "truck": 4, "tract": 4, "tractor": 4, "lorry": 4, "camion": 4,
}

with open(DATASET_DIR / "data.yaml", "r") as f:
    cfg = yaml.safe_load(f)

nombres_orig = cfg["names"]
if isinstance(nombres_orig, list):
    nombres_orig = {i: n for i, n in enumerate(nombres_orig)}

mapa_ids = {
    int(old_id): TRADUCCION[str(nombre).lower().strip()]
    for old_id, nombre in nombres_orig.items()
    if str(nombre).lower().strip() in TRADUCCION
}
print("Clases conservadas:", {nombres_orig[k]: v for k, v in mapa_ids.items()})

for split in ["train", "valid", "val", "test"]:
    labels_dir = DATASET_DIR / split / "labels"
    if not labels_dir.exists():
        continue
    for txt in labels_dir.glob("*.txt"):
        lineas_ok = []
        for linea in txt.read_text().splitlines():
            partes = linea.strip().split()
            if partes and int(partes[0]) in mapa_ids:
                partes[0] = str(mapa_ids[int(partes[0])])
                lineas_ok.append(" ".join(partes))
        txt.write_text("\n".join(lineas_ok) + ("\n" if lineas_ok else ""))

nuevo_yaml = {
    "path": str(DATASET_DIR),
    "train": "train/images",
    "val": "valid/images" if (DATASET_DIR / "valid").exists() else "val/images",
    "names": {0: "moto", 1: "auto", 2: "minibus", 3: "bus", 4: "camion"},
}

DATA_M2_PATH = DATASET_DIR / "data_m2.yaml"
with open(DATA_M2_PATH, "w") as f:
    yaml.safe_dump(nuevo_yaml, f, sort_keys=False)

print("Archivo listo:", DATA_M2_PATH)

Dataset encontrado en: /content/vehicle-counting-20
Clases conservadas: {'auto': 1, 'bike': 0, 'bus': 3, 'car': 1, 'jeep': 1, 'tractor': 4, 'truck': 4}
Archivo listo: /content/vehicle-counting-20/data_m2.yaml


In [ ]:
import shutil
import torch
from pathlib import Path
from ultralytics import YOLO
from google.colab import files

assert torch.cuda.is_available(), "La GPU sigue desactivada. Revisa Entorno de ejecución -> GPU T4."
print("Entrenando en:", torch.cuda.get_device_name(0))

model = YOLO("yolo11s-seg.pt")

model.train(
    data=str(DATA_M2_PATH),
    epochs=40,
    imgsz=640,
    batch=16,
    patience=10,
    device=0,
    project="runs/segment",
    name="m2_vehiculos",
    exist_ok=True,
)

shutil.copy(Path("runs/segment/m2_vehiculos/weights/best.pt"), Path("m2_vehiculos.pt"))
files.download("m2_vehiculos.pt")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Entrenando en: Tesla T4
Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/vehicle-counting-20/data_m2.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0,

FileNotFoundError: [Errno 2] No such file or directory: 'runs/segment/m2_vehiculos/weights/best.pt'

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
shutil.copy("runs/segment/m2_vehiculos/weights/best.pt", "/content/drive/MyDrive/m2_vehiculos.pt")
print("Guardado en Google Drive: MyDrive/m2_vehiculos.pt")

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: 'runs/segment/m2_vehiculos/weights/best.pt'

In [ ]:
import shutil
from pathlib import Path
from google.colab import files, drive

ruta_real = Path("/content/runs/segment/runs/segment/m2_vehiculos/weights/best.pt")
assert ruta_real.exists(), "No se encontró best.pt; revisa que no se haya reiniciado Colab."

# 1. Copiar con el nombre exacto que espera FastAPI
shutil.copy(ruta_real, Path("/content/m2_vehiculos.pt"))

# 2. Guardar respaldo permanente en tu Google Drive
drive.mount("/content/drive")
shutil.copy(ruta_real, Path("/content/drive/MyDrive/m2_vehiculos.pt"))
print("Respaldo guardado en Google Drive: MyDrive/m2_vehiculos.pt")

# 3. Descargar directo a tu PC
files.download("/content/m2_vehiculos.pt")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Respaldo guardado en Google Drive: MyDrive/m2_vehiculos.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import random
import shutil
from pathlib import Path
from google.colab import files

yamls = [p for p in Path("/content").rglob("data_m2.yaml")]
DATASET_M2 = yamls[0].parent
img_dir = DATASET_M2 / "train" / "images"
lbl_dir = DATASET_M2 / "train" / "labels"

pesados, livianos = [], []

for txt in lbl_dir.glob("*.txt"):
    clases = {int(line.split()[0]) for line in txt.read_text().splitlines() if line.strip()}
    img_match = list(img_dir.glob(f"{txt.stem}.*"))
    if not img_match:
        continue
    if 3 in clases or 4 in clases:
        pesados.append(img_match[0])
    elif clases:
        livianos.append(img_match[0])

seleccion = random.sample(pesados, min(150, len(pesados))) + random.sample(livianos, min(100, len(livianos)))
random.shuffle(seleccion)

out_dir = Path("/content/muestra_m3")
out_dir.mkdir(exist_ok=True)
for img in seleccion:
    shutil.copy(img, out_dir / img.name)

shutil.make_archive("/content/muestra_m3", "zip", out_dir)
print(f"Listo: {len(seleccion)} imágenes empaquetadas ({min(150, len(pesados))} pesados, {min(100, len(livianos))} livianos).")
files.download("/content/muestra_m3.zip")

In [ ]:
import random
import shutil
from pathlib import Path
from google.colab import files

yamls = list(Path("/content").rglob("data_m2.yaml"))
DATASET_M2 = yamls[0].parent
img_dir = DATASET_M2 / "train" / "images"
lbl_dir = DATASET_M2 / "train" / "labels"

pesados, livianos = [], []

for txt in lbl_dir.glob("*.txt"):
    clases = {int(line.split()[0]) for line in txt.read_text().splitlines() if line.strip()}
    img_match = list(img_dir.glob(f"{txt.stem}.*"))
    if not img_match:
        continue
    if 3 in clases or 4 in clases:
        pesados.append(img_match[0])
    elif clases:
        livianos.append(img_match[0])

seleccion = random.sample(pesados, min(150, len(pesados))) + random.sample(livianos, min(100, len(livianos)))
random.shuffle(seleccion)

out_dir = Path("/content/muestra_m3")
out_dir.mkdir(exist_ok=True)
for img in seleccion:
    shutil.copy(img, out_dir / img.name)

shutil.make_archive("/content/muestra_m3", "zip", out_dir)
print(f"Listo: {len(seleccion)} imágenes empaquetadas.")
files.download("/content/muestra_m3.zip")

Listo: 250 imágenes empaquetadas.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>